# Biopython - Day 12

In [7]:
from Bio import Entrez, SeqIO
from Bio.SeqUtils import gc_fraction

Entrez.email = "adilsaleem105@gmail.com"  # NCBI requires a real email for API courtesy

# Search for the E. coli acrR gene — one of the genes in your own norfloxacin-resistance manuscript
handle = Entrez.esearch(
    db="nucleotide",
    term="acrR[Gene] AND Escherichia coli O157:H7[Organism] AND RefSeq[Filter]",
    retmax=1
)
record = Entrez.read(handle)
handle.close()

gene_id = record["IdList"][0]
handle = Entrez.efetch(db="nucleotide", id=gene_id, rettype="fasta", retmode="text")
seq_record = SeqIO.read(handle, "fasta")
handle.close()

print("Sequence ID:", seq_record.id)
print("Description:", seq_record.description)
print("Length:", len(seq_record.seq))
print("GC content: {:.2f}%".format(gc_fraction(seq_record.seq) * 100))

Sequence ID: NZ_JAHCTZ010000003.1
Description: NZ_JAHCTZ010000003.1 Escherichia coli O157:H7 strain 438/99 Ec438_99_3, whole genome shotgun sequence
Length: 3297399
GC content: 50.65%


In [27]:
from Bio import Entrez, SeqIO

Entrez.email = "adilsaleem105@gmail.com"  # NCBI requires a real email

ECOLI_O157H7_ACCESSION = "NC_002695.2"  # O157:H7 str. Sakai, complete genome

def fetch_gene_fasta(gene_name, organism="Escherichia coli O157:H7", accession=ECOLI_O157H7_ACCESSION):
    """
    Look up a gene by symbol in NCBI's Gene database (strain-specific),
    then fetch ONLY that gene's sequence slice from the genome — no full
    genome download or manual feature scanning required.
    """
    # 1. Find the strain-specific Gene ID
    search_term = f"{gene_name}[sym] AND {organism}[Organism]"
    handle = Entrez.esearch(db="gene", term=search_term, retmax=5)
    result = Entrez.read(handle)
    handle.close()

    if not result["IdList"]:
        raise ValueError(f"No Gene DB entry for '{gene_name}' in {organism}")

    gene_id = result["IdList"][0]

    # 2. Get genomic coordinates for this gene from its Gene record
    handle = Entrez.efetch(db="gene", id=gene_id, rettype="gene_table", retmode="text")
    # We actually need structured coordinates -> use the docsum/XML route instead:
    handle.close()

    handle = Entrez.esummary(db="gene", id=gene_id)
    summary = Entrez.read(handle)
    handle.close()

    genomic_info = summary["DocumentSummarySet"]["DocumentSummary"][0]["GenomicInfo"][0]
    chr_accver = genomic_info["ChrAccVer"]
    start = int(genomic_info["ChrStart"])
    stop = int(genomic_info["ChrStop"])

    # NCBI gives 0-based coords; efetch seq_start/seq_stop want 1-based, and
    # handles reverse strand automatically if start > stop
    strand = 1 if start <= stop else 2
    seq_start = min(start, stop) + 1
    seq_stop = max(start, stop) + 1

    # 3. Fetch just that slice, directly as FASTA
    handle = Entrez.efetch(
        db="nucleotide",
        id=chr_accver,
        rettype="fasta",
        retmode="text",
        seq_start=seq_start,
        seq_stop=seq_stop,
        strand=strand
    )
    seq_record = SeqIO.read(handle, "fasta")
    handle.close()

    seq_record.id = f"{gene_name}|{chr_accver}:{seq_start}-{seq_stop}"
    seq_record.description = f"{gene_name} ({organism})"
    return seq_record


# --- Run for all four manuscript genes ---
from Bio.SeqUtils import gc_fraction

for gene in ["acrR", "parE", "ompR", "yjgA"]:
    try:
        rec = fetch_gene_fasta(gene)
        print("Gene:", gene)
        print("Sequence ID:", rec.id)
        print("Length:", len(rec.seq))
        print("GC content: {:.2f}%".format(gc_fraction(rec.seq) * 100))
        print()
    except ValueError as e:
        print(f"[!] {gene}: {e}\n")

Gene: acrR
Sequence ID: acrR|NC_002695.2:551827-552474
Length: 648
GC content: 44.91%

Gene: parE
Sequence ID: parE|NC_002695.2:3919726-3921618
Length: 1893
GC content: 53.94%

Gene: ompR
Sequence ID: ompR|NC_002695.2:4248187-4248906
Length: 720
GC content: 54.86%

Gene: yjgA
Sequence ID: yjgA|NC_002695.2:5312921-5313472
Length: 552
GC content: 52.90%



In [ ]:
# Saving the files 


In [28]:
from Bio import SeqIO
import os

os.makedirs("gene_sequences", exist_ok=True)  # creates folder if it doesn't exist

def fetch_and_save_gene(gene_name, organism="Escherichia coli O157:H7", accession=ECOLI_O157H7_ACCESSION):
    rec = fetch_gene_fasta(gene_name, organism, accession)
    filepath = f"gene_sequences/{gene_name}.fasta"
    SeqIO.write(rec, filepath, "fasta")
    print(f"Saved {gene_name} -> {filepath}")
    return rec

for gene in ["acrR", "parE", "ompR", "yjgA"]:
    fetch_and_save_gene(gene)

Saved acrR -> gene_sequences/acrR.fasta
Saved parE -> gene_sequences/parE.fasta
Saved ompR -> gene_sequences/ompR.fasta
Saved yjgA -> gene_sequences/yjgA.fasta


In [29]:
#Saving all the fasta sequence in one file instead of single files like above. 

records = [fetch_gene_fasta(g) for g in ["acrR", "parE", "ompR", "yjgA"]]
SeqIO.write(records, "gene_sequences/manuscript_genes.fasta", "fasta")

4

###  locking on Sakai O157H7 reference genome. 
- Right now the risk is that esummary on the Gene database might, in principle, match a Gene ID annotated on a different O157:H7 sub-strain (e.g., EDL933 or TW14359) if NCBI has multiple gene records under the same organism label. Since your accession NC_002695.2 is Sakai, the safest fix is to verify the returned chr_accver actually matches Sakai's accession before proceeding — and fail loudly if it doesn't:
- What changed: instead of blindly taking result["IdList"][0], it now loops through every candidate Gene ID, checks each one's chr_accver against your target accession (NC_002695, ignoring the version suffix so .2 vs .3 doesn't matter), and only proceeds with the one that's actually on Sakai. If none match, it raises a clear error rather than silently returning the wrong strain's sequence.

In [33]:
from Bio import Entrez, SeqIO
import socket

Entrez.email = "your_email@example.com"
socket.setdefaulttimeout(15)  # force any hung network call to fail after 15 sec instead of forever

ECOLI_O157H7_ACCESSION = "NC_002695.2"

def fetch_gene_fasta(gene_name, organism="Escherichia coli O157:H7", accession=ECOLI_O157H7_ACCESSION):
    print(f"[1/4] Searching Gene DB for '{gene_name}'...")
    search_term = f"{gene_name}[sym] AND {organism}[Organism]"
    handle = Entrez.esearch(db="gene", term=search_term, retmax=20)
    result = Entrez.read(handle)
    handle.close()
    print(f"      Found {len(result['IdList'])} candidate Gene ID(s): {result['IdList']}")

    if not result["IdList"]:
        raise ValueError(f"No Gene DB entry for '{gene_name}' in {organism}")

    for gene_id in result["IdList"]:
        print(f"[2/4] Checking Gene ID {gene_id}...")
        handle = Entrez.esummary(db="gene", id=gene_id)
        summary = Entrez.read(handle)
        handle.close()

        genomic_info = summary["DocumentSummarySet"]["DocumentSummary"][0]["GenomicInfo"][0]
        chr_accver = genomic_info["ChrAccVer"]
        print(f"      -> on accession {chr_accver}")

        if chr_accver.split(".")[0] == accession.split(".")[0]:
            start = int(genomic_info["ChrStart"])
            stop = int(genomic_info["ChrStop"])
            strand = 1 if start <= stop else 2
            seq_start = min(start, stop) + 1
            seq_stop = max(start, stop) + 1
            print(f"[3/4] Match found. Fetching FASTA slice {seq_start}-{seq_stop} (strand {strand})...")

            handle = Entrez.efetch(
                db="nucleotide", id=chr_accver, rettype="fasta", retmode="text",
                seq_start=seq_start, seq_stop=seq_stop, strand=strand
            )
            seq_record = SeqIO.read(handle, "fasta")
            handle.close()

            seq_record.id = f"{gene_name}|{chr_accver}:{seq_start}-{seq_stop}"
            seq_record.description = f"{gene_name} (Sakai, {accession})"
            print(f"[4/4] Done: {gene_name} -> {len(seq_record.seq)} bp\n")
            return seq_record

    raise ValueError(f"'{gene_name}' found in Gene DB but no match on accession {accession} (Sakai)")


# test with just one gene first
rec = fetch_gene_fasta("acrR")
print(rec.id, len(rec.seq))

[1/4] Searching Gene DB for 'acrR'...
      Found 4 candidate Gene ID(s): ['914621', '8215097', '6969188', '957440']
[2/4] Checking Gene ID 914621...
      -> on accession NC_002695.2
[3/4] Match found. Fetching FASTA slice 551827-552474 (strand 1)...
[4/4] Done: acrR -> 648 bp

acrR|NC_002695.2:551827-552474 648


In [34]:
import os
os.makedirs("gene_sequences", exist_ok=True)

records = []
for gene in ["acrR", "parE", "ompR", "yjgA"]:
    rec = fetch_gene_fasta(gene)
    records.append(rec)

# one combined file
from Bio import SeqIO
SeqIO.write(records, "gene_sequences/manuscript_genes_sakai.fasta", "fasta")
print("Saved all 4 genes to gene_sequences/manuscript_genes_sakai.fasta")

[1/4] Searching Gene DB for 'acrR'...
      Found 4 candidate Gene ID(s): ['914621', '8215097', '6969188', '957440']
[2/4] Checking Gene ID 914621...
      -> on accession NC_002695.2
[3/4] Match found. Fetching FASTA slice 551827-552474 (strand 1)...
[4/4] Done: acrR -> 648 bp

[1/4] Searching Gene DB for 'parE'...
      Found 4 candidate Gene ID(s): ['916252', '8218575', '6970381', '958226']
[2/4] Checking Gene ID 916252...
      -> on accession NC_002695.2
[3/4] Match found. Fetching FASTA slice 3919726-3921618 (strand 2)...
[4/4] Done: parE -> 1893 bp

[1/4] Searching Gene DB for 'ompR'...
      Found 4 candidate Gene ID(s): ['915896', '8218921', '6968395', '957766']
[2/4] Checking Gene ID 915896...
      -> on accession NC_002695.2
[3/4] Match found. Fetching FASTA slice 4248187-4248906 (strand 2)...
[4/4] Done: ompR -> 720 bp

[1/4] Searching Gene DB for 'yjgA'...
      Found 4 candidate Gene ID(s): ['913871', '8219900', '6972006', '959807']
[2/4] Checking Gene ID 913871...
     

In [35]:
import os

def fetch_and_save_gene(gene_name, organism="Escherichia coli O157:H7", accession=ECOLI_O157H7_ACCESSION):
    """Fetch one gene's FASTA sequence (Sakai-locked) and save it to its own file."""
    rec = fetch_gene_fasta(gene_name, organism, accession)

    os.makedirs("gene_sequences", exist_ok=True)
    filepath = f"gene_sequences/{gene_name}.fasta"

    from Bio import SeqIO
    SeqIO.write(rec, filepath, "fasta")
    print(f"Saved {gene_name} -> {filepath}")

    return rec


# Example: just change the gene name here
rec = fetch_and_save_gene("acrR")

[1/4] Searching Gene DB for 'acrR'...
      Found 4 candidate Gene ID(s): ['914621', '8215097', '6969188', '957440']
[2/4] Checking Gene ID 914621...
      -> on accession NC_002695.2
[3/4] Match found. Fetching FASTA slice 551827-552474 (strand 1)...
[4/4] Done: acrR -> 648 bp

Saved acrR -> gene_sequences/acrR.fasta


## How to use it going forward:

- Change only the string in fetch_and_save_gene("acrR") to whatever gene you need — e.g. fetch_and_save_gene("recA").
- It depends on fetch_gene_fasta() and ECOLI_O157H7_ACCESSION already being defined earlier in your notebook (from the Sakai-locked version), so make sure that cell has run first.
 -Each call creates one file: gene_sequences/<gene_name>.fasta. If you run it again for the same gene later, it'll just overwrite that file with a fresh fetch.
- rec still holds the sequence in memory too, so you can immediately do things like len(rec.seq) or gc_fraction(rec.seq) right after saving, without needing to re-read the file.

- This keeps your Day 12 "batch of 4 genes" logic and your future "just one gene" use case cleanly separate — same underlying fetch function, two different wrappers depending on what you need that day.